# Highest-Revenue Museum or Related Organization (Most Recent Year)

Self-contained: only requires `museums.csv` in the same folder. Run **Kernel → Restart Kernel and Run All Cells**.

**Approach**
1. Clean the Revenue column (strip sentinel zeros, ensure numeric).
2. Parse Tax Period to a year and find the most recent year available.
3. Deduplicate by EIN (parent organizations share revenue across multiple museums).
4. Identify the top organization and display name, location, type, and revenue.

## Setup

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("museums.csv", low_memory=False)
print(f"Loaded {len(df):,} rows")

Loaded 33,072 rows


## Step 1: Clean the Revenue column

Inspect the column, then convert sentinel zeros to NaN. About 32% of rows show exactly `0` for Revenue — these come from museums that didn't file a 990 or filed a 990-N postcard (which omits dollar amounts). A literal $0 of revenue is implausible for an operating museum.

In [2]:
print(f"dtype          : {df['Revenue'].dtype}")
print(f"missing (NaN)  : {df['Revenue'].isna().sum():,}")
print(f"exact zero     : {(df['Revenue'] == 0).sum():,}")
print(f"negative values: {(df['Revenue'] < 0).sum():,}  (legitimate Form 990 net losses)")
print(f"positive values: {(df['Revenue'] > 0).sum():,}")
print(f"max            : ${df['Revenue'].max():,.0f}")

dtype          : float64
missing (NaN)  : 10,782
exact zero     : 10,783
negative values: 40  (legitimate Form 990 net losses)
positive values: 11,467
max            : $5,840,349,457


In [3]:
# Convert sentinel zeros to NaN; keep the column numeric
df["Revenue"] = df["Revenue"].where(df["Revenue"] != 0, np.nan)
df["Revenue"] = pd.to_numeric(df["Revenue"], errors="coerce")
print(f"Usable revenue values after cleaning: {df['Revenue'].notna().sum():,}")

Usable revenue values after cleaning: 11,507


## Step 2: Identify the most recent year

Tax Period is stored as `YYYYMM` (the fiscal year ending date). Extract the year.

In [4]:
df["tax_year"] = (df["Tax Period"] // 100).astype("Int64")
year_dist = df["tax_year"].value_counts().sort_index()
print("Year distribution:")
print(year_dist.to_string())

most_recent = int(df["tax_year"].max())
print(f"\nMost recent year available: {most_recent}")

Year distribution:
tax_year
1999        1
2002        1
2004        2
2005        2
2007        6
2008       25
2009       22
2010       43
2011      220
2012      645
2013    12218
2014     9968
2015      127

Most recent year available: 2015


## Step 3: Filter to the most recent year and deduplicate by EIN

Many museums share an EIN (Employer Identification Number) with their parent organization — Harvard's 20 museums all file under the same 990 — so the same revenue figure is repeated on many rows. We deduplicate by EIN to get one row per filing entity.

In [5]:
year_df = df[(df["tax_year"] == most_recent) & df["Revenue"].notna()]
print(f"{most_recent} rows with revenue: {len(year_df):,}")

# Keep one row per EIN (revenue is the same across all rows sharing an EIN)
unique_orgs = year_df.drop_duplicates(subset=["Employer ID Number"])
print(f"After dedup by EIN  : {len(unique_orgs):,} unique filing organizations")

2015 rows with revenue: 8
After dedup by EIN  : 7 unique filing organizations


## Step 4: Top 5 organizations

In [6]:
top5 = unique_orgs.nlargest(5, "Revenue")[[
    "Museum Name", "Legal Name",
    "City (Administrative Location)", "State (Administrative Location)",
    "Museum Type", "Revenue"]]
top5.style.format({"Revenue": "${:,.0f}"})

,Museum Name,Legal Name,City (Administrative Location),State (Administrative Location),Museum Type,Revenue
12598,PORTUGUESE AMERICAN WAR VETERANS POST 1,PORTUGUESE AMERICAN WAR VETERANS POST 1,PEABODY,MA,HISTORIC PRESERVATION,"$226,856"
23490,VERMILION-ON-THE-LAKE,VERMILLION-ON-THE-LAKE HISTORIC COMMUNITY CENTER CHARITABLE,VERMILION,OH,HISTORIC PRESERVATION,"$87,082"
5456,BRIDGEVILLE HISTORICAL SOCIETY,BRIDGEVILLE HISTORICAL SOCIETY INC,BRIDGEVILLE,DE,HISTORIC PRESERVATION,"$30,047"
3828,TRINIDAD MUSEUM SOCIETY,TRINIDAD MUSEUM SOCIETY,TRINIDAD,CA,GENERAL MUSEUM,"$22,524"
32117,JACKSON COUNTY HISTORICAL SOCIETY,JACKSON COUNTY HISTORICAL SOCIETY,BLK RIVER FLS,WI,HISTORIC PRESERVATION,"$18,253"


## Step 5: Final answer

In [7]:
winner = unique_orgs.loc[unique_orgs["Revenue"].idxmax()]

print("=" * 70)
print(f"  HIGHEST-REVENUE ORGANIZATION (most recent year, {0})".format(most_recent))
print("=" * 70)
print(f"  Museum Name : {winner['Museum Name']}")
print(f"  Legal Name  : {winner['Legal Name']}")
print(f"  City        : {winner['City (Administrative Location)'].title()}")
print(f"  State       : {winner['State (Administrative Location)']}")
print(f"  Museum Type : {winner['Museum Type']}")
print(f"  Revenue     : ${winner['Revenue']:,.0f}")
print("=" * 70)

  HIGHEST-REVENUE ORGANIZATION (most recent year, 0)
  Museum Name : PORTUGUESE AMERICAN WAR VETERANS POST 1
  Legal Name  : PORTUGUESE AMERICAN WAR VETERANS POST 1
  City        : Peabody
  State       : MA
  Museum Type : HISTORIC PRESERVATION
  Revenue     : $226,856


## Bonus: Same analysis for 2014 (year with bulk of data)

2015 is technically the most recent year, but it has only 127 records — these are organizations with unusual fiscal calendars or late filings. 2014 has the bulk of "recent" data (~10,000 records). For context, here's what 2014 looks like.

In [8]:
y2014 = df[(df["tax_year"] == 2014) & df["Revenue"].notna()]
y2014_unique = y2014.drop_duplicates(subset=["Employer ID Number"])
top5_2014 = y2014_unique.nlargest(5, "Revenue")[[
    "Museum Name", "Legal Name",
    "City (Administrative Location)", "State (Administrative Location)",
    "Museum Type", "Revenue"]]
print(f"2014: {len(y2014_unique):,} unique organizations with revenue\n")
top5_2014.style.format({"Revenue": "${:,.0f}"})

2014: 3,257 unique organizations with revenue



,Museum Name,Legal Name,City (Administrative Location),State (Administrative Location),Museum Type,Revenue
5960,GIFFORD ARBORETUM UNIVERSITY OF MIAMI,UNIVERSITY OF MIAMI,CORAL GABLES,FL,"ARBORETUM, BOTANICAL GARDEN, OR NATURE CENTER","$3,000,690,615"
12416,MCMULLEN MUSEUM OF ART,BOSTON COLLEGE TRUSTEES,CHESTNUT HILL,MA,ART MUSEUM,"$968,788,877"
20503,DR. M.T. GEOFFERY YEH ART GALLERY,ST JOHNS UNIVERSITY,QUEENS,NY,ART MUSEUM,"$855,479,059"
27780,ARMSTRONG BROWNING LIBRARY AND MUSEUM,BAYLOR UNIVERSITY,WACO,TX,GENERAL MUSEUM,"$840,046,127"
8500,ART MUSEUM,DEPAUL UNIVERSITY,CHICAGO,IL,ART MUSEUM,"$757,860,188"


## Summary

**Strict answer (most recent year = 2015):**
- **Portuguese American War Veterans Post 1** — Peabody, MA — Historic Preservation — **$226,856**

This is the maximum revenue among the 127 organizations whose fiscal year ended in 2015. It's a modest figure because 2015 captures only late or unusually-calendared filers; the bulk of post-2013 data sits in 2014.

**For context, 2014 (the most recent year with broad coverage):**
- The top organization by revenue is **University of Miami** (filing for the Gifford Arboretum) at **$3.00 billion** — but this is the *parent university's* total institutional revenue, not the arboretum's own. The same dynamic dominates the top of every year: parent universities, not standalone museums.

**Caveat on "organizational" revenue.** The IMLS dataset's revenue field reflects the IRS Form 990 filing of whatever legal entity owns the museum. For university-owned museums, that's the entire university. The dataset's Legal Name field exposes this — "President and Fellows of Harvard College," "University of Miami," etc. If you need standalone-museum revenue, this dataset isn't the right source; you'd need to filter to entities where the museum is its own 501(c)(3) (e.g., Met Museum, Smithsonian-affiliated, Field Museum).